## Processing large files from pubtator3
*OpenAI used as basis*

In [2]:
import sys
print(sys.executable)
!{sys.executable} -m pip install lxml
!{sys.executable} -m pip install tqdm

/modules/opt/linux-ubuntu24.04-x86_64/jupyterlab/unity-jupyterlab4.4.3/bin/python
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
!which pip
!pip --version

/modules/opt/linux-ubuntu24.04-x86_64/jupyterlab/unity-jupyterlab4.4.3/bin/pip
pip 25.1.1 from /modules/opt/linux-ubuntu24.04-x86_64/jupyterlab/unity-jupyterlab4.4.3/lib/python3.13/site-packages/pip (python 3.13)


In [4]:
!pip install lxml

Defaulting to user installation because normal site-packages is not writeable


In [8]:
import requests
import tarfile
import gzip 
import io
import json
from lxml import etree #both of these should be fine but aren't working 
from tqdm.auto import tqdm #"no module named"
import time
import sqlite3
from typing import Optional
import os
import sys

In [66]:
# Configuration — change as needed before running
URL = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.0.tar.gz"
SQLITE_PATH = "pubtator_bioc0.sqlite"
MEMBER_LIMIT = 12           # set to None to process all members
DOCS_COMMIT_BATCH = 200     # commit every N documents
PROCESS_ONLY_XML = True     # if True, skip non-XML members
SHOW_PROGRESS = True

In [49]:
# %% [markdown]
# SQLite schema and initialization
CREATE_TABLES_SQL = """
PRAGMA journal_mode = WAL;
PRAGMA synchronous = NORMAL;

CREATE TABLE IF NOT EXISTS documents (
    doc_id TEXT PRIMARY KEY,
    member_name TEXT,        -- which archive member the document came from
    infons TEXT,             -- JSON
    title TEXT,
    abstract TEXT
);

CREATE TABLE IF NOT EXISTS passages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_id TEXT,
    member_name TEXT,
    offset INTEGER,
    infons TEXT,
    text TEXT
);

CREATE TABLE IF NOT EXISTS sentences (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    passage_id INTEGER,
    infons TEXT,
    text TEXT
);

CREATE TABLE IF NOT EXISTS annotations (
    ann_id TEXT PRIMARY KEY,
    doc_id TEXT,
    passage_id INTEGER,
    sentence_id INTEGER,
    location_offset INTEGER,
    location_length INTEGER,
    text TEXT,
    infons TEXT
);

CREATE TABLE IF NOT EXISTS relations (
    rel_id TEXT PRIMARY KEY,
    doc_id TEXT,
    passage_id INTEGER,
    sentence_id INTEGER,
    infons TEXT
);

CREATE TABLE IF NOT EXISTS relation_nodes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    rel_id TEXT,
    refid TEXT,
    role TEXT
);

-- bookkeeping: record processed member names so we can resume
CREATE TABLE IF NOT EXISTS processed_members (
    member_name TEXT PRIMARY KEY,
    processed_at REAL
);

CREATE INDEX IF NOT EXISTS idx_passages_doc ON passages(doc_id);
CREATE INDEX IF NOT EXISTS idx_annotations_doc ON annotations(doc_id);
"""


In [35]:
def init_db(path: str):
    conn = sqlite3.connect(path, timeout=60)
    cur = conn.cursor()
    cur.executescript(CREATE_TABLES_SQL)
    conn.commit()
    return conn

In [36]:
# Small XML helper utilities
def localname(tag: Optional[str]) -> Optional[str]:
    if tag is None:
        return None
    return tag.split("}")[-1] if "}" in tag else tag

def get_child_text(elem, child_name: str) -> Optional[str]:
    for ch in elem:
        if localname(ch.tag) == child_name:
            return (ch.text or "").strip()
    return None

def get_children(elem, child_name: str):
    for ch in elem:
        if localname(ch.tag) == child_name:
            yield ch

def infons_to_dict(parent_elem):
    d = {}
    for inf in get_children(parent_elem, "infon"):
        if "key" in inf.attrib:
            key = inf.attrib["key"]
            d[key] = (inf.text or "").strip()
        else:
            # fallback: accumulate unnamed infons
            v = (inf.text or "").strip()
            if v:
                d.setdefault("notes", []).append(v)
    return d

In [63]:
# Core: parse one archive member (BioC XML) incrementally and insert into DB
def parse_member_and_insert(member_fileobj, member_name: str, conn: sqlite3.Connection,
                            docs_commit_batch: int = DOCS_COMMIT_BATCH,
                            max_docs: Optional[int]=None,
                            show_progress: bool = SHOW_PROGRESS):
    """
    member_fileobj: binary file-like for the member bytes
    member_name: name string (used for bookkeeping)
    conn: sqlite connection
    max_docs: optional limit of documents to process for this member (useful for testing)
    Returns: number of documents processed
    """
    # wrap gzip if member itself is gz inside the tar
    if member_name.endswith(".gz"):
        binstream = gzip.GzipFile(fileobj=member_fileobj)
    else:
        binstream = member_fileobj

    parser = etree.XMLParser(recover=True, huge_tree=True)
    context = etree.iterparse(binstream, events=("end",))

    cur = conn.cursor()
    docs = 0
    t0 = time.time()

    try:
        for event, elem in context:
            #print("localname(elem.tag):", localname(elem.tag))
            if localname(elem.tag) and localname(elem.tag).lower() == "document":
                # document id
                doc_id = get_child_text(elem, "id") or get_child_text(elem, "pmid") or None
                doc_infons = infons_to_dict(elem)
                title_text = None
                abstract_text = None

                # insert document row
                cur.execute(
                    "INSERT OR REPLACE INTO documents(doc_id, member_name, infons, title, abstract) VALUES (?, ?, ?, ?, ?)",
                    (doc_id, member_name, json.dumps(doc_infons, ensure_ascii=False), None, None)
                )

                # process passages
                for passage in (ch for ch in elem if localname(ch.tag) == "passage"):
                    p_infons = infons_to_dict(passage)
                    # offset
                    offset = None
                    off = get_child_text(passage, "offset")
                    if off and off.isdigit():
                        offset = int(off)
                    else:
                        for loc in get_children(passage, "location"):
                            if "offset" in loc.attrib:
                                try:
                                    offset = int(loc.attrib.get("offset"))
                                except Exception:
                                    pass
                                break
                    p_text = get_child_text(passage, "text") or ""

                    cur.execute(
                        "INSERT INTO passages(doc_id, member_name, offset, infons, text) VALUES (?, ?, ?, ?, ?)",
                        (doc_id, member_name, offset, json.dumps(p_infons, ensure_ascii=False), p_text)
                    )
                    passage_rowid = cur.lastrowid

                    # heuristics for title/abstract
                    ptype = p_infons.get("type") or p_infons.get("section")
                    if ptype:
                        if ptype.lower() in ("title", "article_title"):
                            title_text = p_text
                        elif ptype.lower() in ("abstract", "abstract_text"):
                            abstract_text = (abstract_text + "\n" + p_text) if abstract_text else p_text

                    # sentences
                    for sentence in (s for s in passage if localname(s.tag) == "sentence"):
                        s_infons = infons_to_dict(sentence)
                        s_text = get_child_text(sentence, "text") or ""
                        print("Found sentence: " + sentence)
                        cur.execute(
                            "INSERT INTO sentences(passage_id, infons, text) VALUES (?, ?, ?)",
                            (passage_rowid, json.dumps(s_infons, ensure_ascii=False), s_text)
                        )
                        sentence_rowid = cur.lastrowid

                        # annotations inside sentence
                        for ann in (a for a in sentence if localname(a.tag) == "annotation"):
                            ann_id = get_child_text(ann, "id") or None
                            ann_text = get_child_text(ann, "text") or None
                            ann_infons = infons_to_dict(ann)
                            loc_offset = None
                            loc_len = None
                            for loc in get_children(ann, "location"):
                                if "offset" in loc.attrib:
                                    try:
                                        loc_offset = int(loc.attrib.get("offset"))
                                    except Exception:
                                        loc_offset = None
                                if "length" in loc.attrib:
                                    try:
                                        loc_len = int(loc.attrib.get("length"))
                                    except Exception:
                                        loc_len = None
                                if loc_offset is None and loc.text and loc.text.isdigit():
                                    loc_offset = int(loc.text)
                            cur.execute(
                                "INSERT OR REPLACE INTO annotations(ann_id, doc_id, passage_id, sentence_id, location_offset, location_length, text, infons) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                                (ann_id, doc_id, passage_rowid, sentence_rowid, loc_offset, loc_len, ann_text, json.dumps(ann_infons, ensure_ascii=False))
                            )

                        # relations inside sentence
                        for rel in (r for r in sentence if localname(r.tag) == "relation"):
                            rel_id = get_child_text(rel, "id") or None
                            rel_infons = infons_to_dict(rel)
                            cur.execute(
                                "INSERT OR REPLACE INTO relations(rel_id, doc_id, passage_id, sentence_id, infons) VALUES (?, ?, ?, ?, ?)",
                                (rel_id, doc_id, passage_rowid, sentence_rowid, json.dumps(rel_infons, ensure_ascii=False))
                            )
                            for node in get_children(rel, "node"):
                                refid = node.get("refid")
                                role = node.get("role")
                                cur.execute(
                                    "INSERT INTO relation_nodes(rel_id, refid, role) VALUES (?, ?, ?)",
                                    (rel_id, refid, role)
                                )

                    # annotations directly under passage
                    for ann in (a for a in passage if localname(a.tag) == "annotation"):
                        ann_id = get_child_text(ann, "id") or None
                        ann_text = get_child_text(ann, "text") or None
                        ann_infons = infons_to_dict(ann)
                        loc_offset = None
                        loc_len = None
                        for loc in get_children(ann, "location"):
                            if "offset" in loc.attrib:
                                try:
                                    loc_offset = int(loc.attrib.get("offset"))
                                except Exception:
                                    loc_offset = None
                            if "length" in loc.attrib:
                                try:
                                    loc_len = int(loc.attrib.get("length"))
                                except Exception:
                                    loc_len = None
                            if loc_offset is None and loc.text and loc.text.isdigit():
                                loc_offset = int(loc.text)
                        cur.execute(
                            "INSERT OR REPLACE INTO annotations(ann_id, doc_id, passage_id, sentence_id, location_offset, location_length, text, infons) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                            (ann_id, doc_id, passage_rowid, None, loc_offset, loc_len, ann_text, json.dumps(ann_infons, ensure_ascii=False))
                        )

                    # relations directly under passage
                    for rel in (r for r in passage if localname(r.tag) == "relation"):
                        rel_id = get_child_text(rel, "id") or None
                        rel_infons = infons_to_dict(rel)
                        cur.execute(
                            "INSERT OR REPLACE INTO relations(rel_id, doc_id, passage_id, sentence_id, infons) VALUES (?, ?, ?, ?, ?)",
                            (rel_id, doc_id, passage_rowid, None, json.dumps(rel_infons, ensure_ascii=False))
                        )
                        for node in get_children(rel, "node"):
                            refid = node.get("refid")
                            role = node.get("role")
                            cur.execute(
                                "INSERT INTO relation_nodes(rel_id, refid, role) VALUES (?, ?, ?)",
                                (rel_id, refid, role)
                            )

                # annotations/relations directly under document
                for ann in (a for a in elem if localname(a.tag) == "annotation"):
                    ann_id = get_child_text(ann, "id") or None
                    ann_text = get_child_text(ann, "text") or None
                    ann_infons = infons_to_dict(ann)
                    loc_offset = None
                    loc_len = None
                    for loc in get_children(ann, "location"):
                        if "offset" in loc.attrib:
                            try:
                                loc_offset = int(loc.attrib.get("offset"))
                            except Exception:
                                loc_offset = None
                        if "length" in loc.attrib:
                            try:
                                loc_len = int(loc.attrib.get("length"))
                            except Exception:
                                loc_len = None
                    cur.execute(
                        "INSERT OR REPLACE INTO annotations(ann_id, doc_id, passage_id, sentence_id, location_offset, location_length, text, infons) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                        (ann_id, doc_id, None, None, loc_offset, loc_len, ann_text, json.dumps(ann_infons, ensure_ascii=False))
                    )

                for rel in (r for r in elem if localname(r.tag) == "relation"):
                        rel_id = get_child_text(rel, "id") or None
                        rel_infons = infons_to_dict(rel)
                        cur.execute(
                            "INSERT OR REPLACE INTO relations(rel_id, doc_id, passage_id, sentence_id, infons) VALUES (?, ?, ?, ?, ?)",
                            (rel_id, doc_id, None, None, json.dumps(rel_infons, ensure_ascii=False))
                        )
                        for node in get_children(rel, "node"):
                            refid = node.get("refid")
                            role = node.get("role")
                            cur.execute(
                                "INSERT INTO relation_nodes(rel_id, refid, role) VALUES (?, ?, ?)",
                                (rel_id, refid, role)
                            )
                            
                if title_text or abstract_text:
                    cur.execute("UPDATE documents SET title = ?, abstract = ? WHERE doc_id = ?", (title_text, abstract_text, doc_id))

                docs += 1
                if docs % docs_commit_batch == 0:
                    conn.commit()
                    if show_progress:
                        elapsed = time.time() - t0
                        print(f"  inserted {docs} docs from member {member_name} (elapsed {elapsed:.1f}s)")

                # free memory
                elem.clear()
                while elem.getprevious() is not None:
                    del elem.getparent()[0]

                if max_docs is not None and docs >= max_docs:
                    break
    finally:
        # final commit for this member
        conn.commit()
        try:
            context.close()
        except Exception:
            pass

    return docs

In [64]:
# Stream the tar archive, process each member, and record progress for resume
def stream_tar_and_process(url: str, conn: sqlite3.Connection,
                           member_limit: Optional[int] = MEMBER_LIMIT,
                           docs_commit_batch: int = DOCS_COMMIT_BATCH,
                           process_only_xml: bool = PROCESS_ONLY_XML,
                           resume: bool = True):
    """
    Streams the tar.gz at url and processes members sequentially.
    If resume=True, members listed in processed_members table are skipped.
    """
    # Get set of already processed members
    cur = conn.cursor()
    if resume:
        cur.execute("SELECT member_name FROM processed_members")
        processed_set = set(r[0] for r in cur.fetchall())
    else:
        processed_set = set()

    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    resp.raw.decode_content = True
    # choose streaming mode; let tarfile detect compression
    tar = tarfile.open(fileobj=resp.raw, mode="r|*")

    members_done = 0
    total_docs = 0
    t0_all = time.time()

    try:
        for member in tar:
            if member_limit is not None and members_done >= member_limit:
                break
            if not member.isfile():
                continue
            name = member.name
            if resume and name in processed_set:
                if SHOW_PROGRESS:
                    print(f"Skipping already-processed member: {name}")
                members_done += 1
                continue
            # Optionally skip non-XML files
            if process_only_xml and not (name.lower().endswith(".xml") or name.lower().endswith(".xml.gz") or name.lower().endswith(".bioc") or name.lower().endswith(".bioc.gz")):
                if SHOW_PROGRESS:
                    print(f"Skipping non-XML member: {name}")
                members_done += 1
                # we don't mark non-XML members as processed to allow future runs to reconsider them
                continue

            if SHOW_PROGRESS:
                print(f"Processing member: {name}")

            fobj = tar.extractfile(member)
            if fobj is None:
                print(f"  [WARN] could not extract {name}")
                members_done += 1
                continue

            try:
                docs = parse_member_and_insert(fobj, name, conn, docs_commit_batch)
            except Exception as e:
                # On errors, commit what we have and raise or continue based on policy.
                conn.commit()
                print(f"  [ERROR] parsing member {name}: {e}", file=sys.stderr)
                # Option: mark as failed by not recording processed_members so you can retry later.
                # We'll re-raise to stop unless you prefer to continue
                raise
            finally:
                try:
                    fobj.close()
                except Exception:
                    pass

            # record that member completed
            cur.execute("INSERT OR REPLACE INTO processed_members(member_name, processed_at) VALUES (?, ?)", (name, time.time()))
            conn.commit()

            members_done += 1
            total_docs += docs
            if SHOW_PROGRESS:
                print(f"  finished member {name}: {docs} documents")

    finally:
        try:
            tar.close()
        except Exception:
            pass
        try:
            resp.close()
        except Exception:
            pass

    elapsed = time.time() - t0_all
    print(f"Completed {members_done} members, {total_docs} documents in {elapsed:.1f}s")

In [67]:
# Main: initialize DB and run the stream processing
if __name__ == "__main__":
    # Create DB if needed
    conn = init_db(SQLITE_PATH)
    try:
        stream_tar_and_process(URL, conn, member_limit=MEMBER_LIMIT)
    finally:
        conn.close()

Skipping already-processed member: output/BioCXML/10.BioC.XML
Skipping already-processed member: output/BioCXML/100.BioC.XML
Skipping already-processed member: output/BioCXML/1000.BioC.XML
Skipping already-processed member: output/BioCXML/10000.BioC.XML
Skipping already-processed member: output/BioCXML/100000.BioC.XML
Skipping already-processed member: output/BioCXML/100010.BioC.XML
Skipping already-processed member: output/BioCXML/100020.BioC.XML
Skipping already-processed member: output/BioCXML/100030.BioC.XML
Skipping already-processed member: output/BioCXML/100040.BioC.XML
Skipping already-processed member: output/BioCXML/100050.BioC.XML
Skipping already-processed member: output/BioCXML/100060.BioC.XML
Processing member: output/BioCXML/100070.BioC.XML
  inserted 200 docs from member output/BioCXML/100070.BioC.XML (elapsed 0.1s)
  inserted 400 docs from member output/BioCXML/100070.BioC.XML (elapsed 0.2s)
  inserted 600 docs from member output/BioCXML/100070.BioC.XML (elapsed 0.2s)


In [14]:
# %% [markdown]
# Quick queries and tips
# - To inspect progress:
#     sqlite3 pubtator_bioc0.sqlite "SELECT COUNT(*) FROM processed_members;"
# - To see total documents imported:
#     sqlite3 pubtator_bioc0.sqlite "SELECT COUNT(*) FROM documents;"
# - If you want a different schema (e.g., normalized annotation types in columns), tell me which infon keys you care about and I can adapt the script to extract them into dedicated columns.
# - If you'd like the script to be more robust to intermittent network failures (auto-retry, resume after transient errors), I can add retry logic and exponential backoff.
#
# Run this notebook locally. Start with MEMBER_LIMIT small (e.g., 1 or 5) to confirm behavior, then set to None to process the entire archive.